In [0]:
# Databricks notebook source

# COMMAND ----------

from functools import reduce
from pyspark.sql import functions as F

# COMMAND ----------

taxi_type = "yellow"
year = 2023
months = [1, 2, 3, 4, 5]

raw_base_path = f"/Volumes/workspace/raw/landing_zone/{taxi_type}_taxi/year={year}"
bronze_table = "workspace.bronze.yellow_taxi_trips"

# COMMAND ----------

def read_month_file(month: int):
    month_str = str(month).zfill(2)

    file_path = (
        f"{raw_base_path}/month={month_str}/"
        f"{taxi_type}_tripdata_{year}-{month_str}.parquet"
    )

    print(f"Lendo arquivo: {file_path}")

    df = spark.read.parquet(file_path)

    # Converte timestamp_ntz para string na Bronze para evitar erro de feature Delta.
    # Na Silver vamos converter novamente para timestamp.
    for col_name, col_type in df.dtypes:
        if col_type == "timestamp_ntz":
            df = df.withColumn(col_name, F.col(col_name).cast("string"))

    df_bronze = (
        df
        .withColumn("source_file_month", F.lit(month_str))
        .withColumn("source_year", F.lit(year))
        .withColumn("source_system", F.lit("nyc_tlc"))
        .withColumn("source_entity", F.lit("yellow_taxi"))
        .withColumn("ingestion_timestamp", F.current_timestamp().cast("string"))
    )

    return df_bronze

# COMMAND ----------

dataframes = []

for month in months:
    df_month = read_month_file(month)
    dataframes.append(df_month)

df_bronze = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    dataframes
)

# COMMAND ----------

display(df_bronze.limit(10))

# COMMAND ----------

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

# COMMAND ----------

display(spark.table(bronze_table).limit(10))

# COMMAND ----------

spark.sql(f"""
SELECT
    source_year,
    source_file_month,
    COUNT(*) AS total_records
FROM {bronze_table}
GROUP BY source_year, source_file_month
ORDER BY source_file_month
""").show()